In [ ]:
'''
A reusable notebook template for member-wise diagnostics
The helpers handle structure, labels, checks, aggregation, and persistence. You supply the scientific calculation. 
The archive is an ordinary dictionary, not a class or framework.
Run the helper cell once. Then edit the setup cell's quantity definitions, experiment choices, mask, save settings, 
and metadata. In the calculation cell, fill here with one numeric result for every registered field and each member. 
The example calculations are deliberately commented out; an unfilled block stops with a clear error instead of inventing results. 
Afterwards aggregate, inspect, and optionally save. The final load-only cell is opt-in and requires no original data.
What each structure contains
- diagnostics["member_values"][experiment][quantity][model]: (member, ...) arrays; initialized to None until computed.
- diagnostics["model_means"][experiment][quantity]: (model, ...), ordered by diagnostics["model_names"][experiment].
- diagnostics["MMM"][experiment][quantity]: (...), with equal weight for each available model.
- model_n_valid and MMM_n_valid: contributor counts matching those two summaries.
- quantity_info, member_ids, coordinates, mask, field_shapes, and metadata: labels and provenance preserved with the values.
A quantity can be a scalar, map, series, matrix, or higher-dimensional real numeric array. Different quantities can have different shapes. 
Within a quantity, all members/models/experiments must have a common per-member shape and scientifically aligned axes; shapes are checked, 
but the helpers cannot infer physical alignment. Ragged arrays, object values, complex arrays, regridding, time alignment, 
or mode matching are not handled automatically.
Use separate archives or explicitly align/pad values yourself when needed.
Scientific choices remain explicit
Aggregation is the finite arithmetic mean of your finished member diagnostic, followed by the equal-model mean. 
It does not decide to square SDs, take square roots, Fisher-transform correlations, or recompute a diagnostic from averaged fields. 
Register those as separate member calculations when desired. NaN/inf entries are omitted from means, with counts retained. 
Entirely undefined numeric results are allowed; uncomputed None slots are not. Model/member availability can differ entry-by-entry; 
there is no automatic cross-experiment common-support filter.
mask=None leaves outputs unchanged. A supplied Boolean or 1/NaN mask is applied only to fields with apply_mask=True, 
and only to their final two latitude/longitude axes when stored. It is not a fractional area weight and does not change your inputs, 
regional averages, regressions, or other calculations upstream. Apply input masks/weights explicitly yourself where scientifically needed. 
Optional dims labels and supplied coordinates check axis lengths, not data-coordinate provenance.
Default groups are historical, historical_matched, and amip_hist. An optional visible reuse step copies historical-matched model/member 
subsets from historical by actual member ID, keeping independent arrays. Disable reuse if the calculation itself depends on the surrounding model population. 
Other projects can supply different experiment/model/member dictionaries.
Saving and reuse
save=False means no writes. savepath is a single .npy filename, not a directory. A completed archive includes all structures and metadata in that file; 
existing files are protected unless overwrite=True. A same-directory temporary file is fully written before publication, 
so a failed write does not replace the completed destination. No-overwrite publication uses a hard link, supported on the local Mac/Linux filesystems targeted here. 
Archives use pickle: load only files you created or trust.
The save helper validates completion and shapes, not scientific correctness or semantic axis alignment. Use store_diagnostic_members to update arrays, 
then rerun aggregation; after any direct dictionary/array edits you are also responsible for rerunning aggregation. Refresh convenience aliases after 
replacing/aggregating the archive. Setup resets this NEW output archive, never your input archive. The default load cell leaves current results alone 
and uses distinct loaded_... names when enabled.
For later notebooks, import the six public functions from diagnostic_archive_helpers.py instead of copying the helper definitions again. 
The companion diagnostic_archive_template_cells.py contains just the five editable workflow cells. To add a diagnostic, 
register its metadata in setup and assign here["new_name"] in the member loop; collection, checking, aggregation, 
and saving then follow the same structure automatically.
'''

In [ ]:
"""Plain-dictionary helpers for notebook diagnostics; no scientific calculations hidden here."""
import numpy as np
from copy import deepcopy
from pathlib import Path
from datetime import datetime, timezone
import os
import tempfile


def setup_diagnostic_archive(quantity_info, model_names, member_ids, *,
                             experiments=("historical", "historical_matched", "amip_hist"),
                             coordinates=None, mask=None, metadata=None,
                             save=False, savepath=None, overwrite=False):
    """Create empty member/model/MMM dictionaries and copy the selected labels.

    quantity_info maps your field names to dictionaries of units/meaning/etc.
    Optional field entries: shape (per member), dims (excluding member), and
    apply_mask=True (use the supplied 2-D mask on the final lat/lon axes).
    Unspecified shapes are learned from the first stored model for each field.
    save/savepath/overwrite are settings ONLY: setup performs no disk writes.
    """
    experiments = list(experiments)
    if not experiments or len(experiments) != len(set(experiments)):
        raise ValueError("experiments must be nonempty and unique.")
    if not quantity_info or any(not isinstance(q, str) or not q.strip() for q in quantity_info):
        raise ValueError("Supply a nonempty quantity_info dictionary with meaningful string keys.")
    info, coords = deepcopy(quantity_info), deepcopy(coordinates or {})
    names, ids, shapes = {}, {}, {}
    for quantity, description in info.items():
        if not isinstance(description, dict):
            raise ValueError(f"{quantity}: metadata must be a dictionary.")
        shape = description.get("shape")
        if shape is not None:
            shape = tuple(shape)
            if any(not isinstance(n, (int, np.integer)) or n < 1 for n in shape):
                raise ValueError(f"{quantity}: shape must contain positive integers; () denotes a scalar.")
        shapes[quantity] = shape
        if description.get("dims") is not None:
            description["dims"] = tuple(description["dims"])
    for experiment in experiments:
        names[experiment] = list(model_names[experiment])
        if not names[experiment] or len(names[experiment]) != len(set(names[experiment])):
            raise ValueError(f"{experiment}: model names must be nonempty and unique.")
        ids[experiment] = {}
        for model in names[experiment]:
            labels = np.asarray(member_ids[experiment][model])
            if labels.ndim != 1 or not len(labels) or len(set(labels.tolist())) != len(labels):
                raise ValueError(f"{experiment}/{model}: supply nonempty, unique 1-D member labels.")
            ids[experiment][model] = labels.copy()

    # A mask is an inclusion rule, not a fractional area weight. It is stored
    # here, but applied only by store_diagnostic_members for opt-in fields.
    selected_mask = None
    if mask is not None:
        supplied = np.ma.asarray(mask, dtype=float).filled(np.nan)
        if supplied.ndim != 2:
            raise ValueError("mask must be a 2-D inclusion mask, or None.")
        selected_mask = np.isfinite(supplied) & (supplied != 0)
        if "lat" in coords and "lon" in coords and selected_mask.shape != (len(coords["lat"]), len(coords["lon"])):
            raise ValueError("mask must match the supplied latitude/longitude grid.")

    # None means NOT YET COMPUTED, rather than a valid zero or an all-NaN result.
    # A field can be a scalar, series, map, matrix, or higher-dimensional array.
    empty_summary = lambda: {e: {q: None for q in info} for e in experiments}
    return {
        "schema": "member_diagnostic_archive", "schema_version": 1,
        "experiments": experiments, "quantity_info": info, "field_shapes": shapes,
        "member_values": {e: {q: {m: None for m in names[e]} for q in info} for e in experiments},
        "model_means": empty_summary(), "MMM": empty_summary(),
        "model_n_valid": empty_summary(), "MMM_n_valid": empty_summary(),
        "model_names": names, "member_ids": ids, "coordinates": coords, "mask": selected_mask,
        "settings": {"save": bool(save), "savepath": None if savepath is None else str(savepath),
                     "overwrite": bool(overwrite)},
        "aggregated": False,
        "metadata": {
            "created_utc": datetime.now(timezone.utc).isoformat(), "user": deepcopy(metadata or {}),
            "aggregation": "Finite arithmetic member means, then equal-model means, entry by entry",
            "missing": "NaN/inf omitted from means; valid counts retained; None means not computed",
            "mask_rule": "Finite nonzero includes; applied only to final two axes of opt-in fields",
            "alignment": "No regridding, time alignment, mode matching, or member pairing is inferred",
            "reused_from": {},
        },
    }


def _checked_member_array(archive, experiment, quantity, model, values):
    """Check one completed model's member array without altering its values."""
    # Looking up this slot also checks experiment/quantity/model membership.
    archive["member_values"][experiment][quantity][model]
    array = np.ma.asarray(values)
    if array.dtype.kind not in "biuf":
        raise ValueError(f"{experiment}/{model}/{quantity}: supply real numeric values, not objects/strings/complex data.")
    array = np.ma.asarray(array, dtype=float).filled(np.nan)
    n_members = len(archive["member_ids"][experiment][model])
    if array.ndim < 1 or array.shape[0] != n_members or any(n == 0 for n in array.shape):
        raise ValueError(f"{experiment}/{model}/{quantity}: expected member axis of length {n_members}; got {array.shape}.")
    shape, expected = array.shape[1:], archive["field_shapes"][quantity]
    if expected is not None and shape != expected:
        raise ValueError(f"{quantity}: per-member shape {shape} differs from registered/inferred {expected}.")
    dims = archive["quantity_info"][quantity].get("dims")
    if dims is not None:
        if len(dims) != len(shape):
            raise ValueError(f"{quantity}: dims must describe the PER-MEMBER axes {shape}.")
        for dimension, size in zip(dims, shape):
            if dimension in archive["coordinates"]:
                coord = np.asarray(archive["coordinates"][dimension])
                if coord.ndim != 1 or len(coord) != size:
                    raise ValueError(f"{quantity}: dimension {dimension} does not match its saved coordinate.")
    return array


def store_diagnostic_members(archive, experiment, quantity, model, values):
    """Store (member, ...) values, checking labels/shapes and applying an opted-in mask.

    The supplied member order must match archive['member_ids'][experiment][model].
    This does not calculate the diagnostic or mask any inputs to that calculation.
    Updating values invalidates summaries; aggregate again before saving.
    """
    array = _checked_member_array(archive, experiment, quantity, model, values)
    description, mask = archive["quantity_info"][quantity], archive["mask"]
    if description.get("apply_mask", False) and mask is not None:
        dims = description.get("dims")
        if array.ndim < 3 or array.shape[-2:] != mask.shape:
            raise ValueError(f"{quantity}: masked fields must end in the mask's (lat, lon) shape.")
        if dims is not None and tuple(dims[-2:]) != ("lat", "lon"):
            raise ValueError(f"{quantity}: apply_mask requires final dimensions ('lat', 'lon').")
        array = np.where(mask, array, np.nan)
    else:
        array = array.copy()       # Later edits to stored data must not alter source arrays.
    archive["member_values"][experiment][quantity][model] = array
    archive["field_shapes"][quantity] = array.shape[1:]
    archive["aggregated"] = False
    # Clear all means to prevent inadvertently inspecting stale aggregated results.
    for key in ["model_means", "MMM", "model_n_valid", "MMM_n_valid"]:
        for e in archive["experiments"]:
            for q in archive["quantity_info"]:
                archive[key][e][q] = None


def copy_diagnostic_subset(archive, source="historical", target="historical_matched", *, overwrite=False):
    """Reuse a subset's member diagnostics without rerunning their calculations.

    Models must exist in source. Members are selected/reordered by explicit IDs,
    not file positions. Copies are independent arrays. Appropriate only when the
    diagnostic itself does not change with the selected experiment/model group.
    """
    if source == target:
        raise ValueError("source and target must be different experiments.")
    selections = {}
    # Check every requested slot before copying, so missing IDs/results do not
    # leave a partially copied target. A populated target needs explicit overwrite.
    for model in archive["model_names"][target]:
        lookup = {label: i for i, label in enumerate(archive["member_ids"][source][model].tolist())}
        labels = archive["member_ids"][target][model].tolist()
        if any(label not in lookup for label in labels):
            raise ValueError(f"{target}/{model}: a member ID is absent from {source}.")
        selections[model] = [lookup[label] for label in labels]
        for quantity in archive["quantity_info"]:
            values = archive["member_values"][source][quantity][model]
            if values is None:
                raise ValueError(f"Calculate {source}/{model}/{quantity} before copying.")
            _checked_member_array(archive, source, quantity, model, values)
            if archive["member_values"][target][quantity][model] is not None and not overwrite:
                raise ValueError(f"{target}/{model}/{quantity} already populated; set overwrite=True deliberately.")
    for model, positions in selections.items():
        for quantity in archive["quantity_info"]:
            values = archive["member_values"][source][quantity][model][positions]
            store_diagnostic_members(archive, target, quantity, model, values)
    archive["metadata"]["reused_from"][target] = source


def _finite_diagnostic_mean(values):
    """Finite mean/count along the first axis; works for scalars through ND arrays."""
    finite = np.isfinite(values)
    count = finite.sum(axis=0)
    total = np.where(finite, values, 0.).sum(axis=0)
    mean = np.divide(total, count, out=np.full_like(total, np.nan, dtype=float), where=count > 0)
    return mean, count


def aggregate_diagnostic_archive(archive):
    """Average completed member diagnostics within models, then equally across models.

    These are literal arithmetic means of YOUR calculated quantities. No Fisher-z,
    squaring, root, pooling, or variance estimation is automatically performed.
    All-NaN results are allowed (with zero counts); uncomputed None slots are not.
    """
    archive["aggregated"] = False
    missing = [(e, q, m) for e in archive["experiments"] for q in archive["quantity_info"]
               for m in archive["model_names"][e] if archive["member_values"][e][q][m] is None]
    if missing:
        raise ValueError(f"{len(missing)} model/quantity slots are uncomputed. First entries: {missing[:5]}")
    summaries = {key: {e: {} for e in archive["experiments"]}
                 for key in ["model_means", "MMM", "model_n_valid", "MMM_n_valid"]}
    for experiment in archive["experiments"]:
        for quantity in archive["quantity_info"]:
            means, counts = [], []
            for model in archive["model_names"][experiment]:
                values = _checked_member_array(archive, experiment, quantity, model,
                                                archive["member_values"][experiment][quantity][model])
                mean, count = _finite_diagnostic_mean(values)
                means.append(mean)
                counts.append(count)
            # Stack in the explicit model order. No model gains weight merely
            # by having more members. Counts vary entry-by-entry when data are missing.
            stack = np.stack(means, axis=0)
            summaries["model_means"][experiment][quantity] = stack
            summaries["model_n_valid"][experiment][quantity] = np.stack(counts, axis=0)
            summaries["MMM"][experiment][quantity], summaries["MMM_n_valid"][experiment][quantity] = _finite_diagnostic_mean(stack)
    archive.update(summaries)
    archive["aggregated"] = True
    archive["metadata"]["aggregated_utc"] = datetime.now(timezone.utc).isoformat()
    return archive


def _check_completed_diagnostic_archive(archive):
    """Check schema, completion, member counts, and summary shapes, not scientific correctness."""
    if not isinstance(archive, dict) or archive.get("schema") != "member_diagnostic_archive" or archive.get("schema_version") != 1:
        raise ValueError("Not a supported member diagnostic archive.")
    if not archive.get("aggregated", False):
        raise ValueError("Aggregate completed diagnostics before saving/loading this archive.")
    for experiment in archive["experiments"]:
        for quantity in archive["quantity_info"]:
            shape = archive["field_shapes"][quantity]
            if shape is None:
                raise ValueError(f"{quantity}: no per-member shape registered.")
            for model in archive["model_names"][experiment]:
                _checked_member_array(archive, experiment, quantity, model,
                                      archive["member_values"][experiment][quantity][model])
            for key in ["model_means", "model_n_valid", "MMM", "MMM_n_valid"]:
                expected = (len(archive["model_names"][experiment]),) + shape if key.startswith("model_") else shape
                if np.shape(archive[key][experiment][quantity]) != expected:
                    raise ValueError(f"{key}/{experiment}/{quantity}: invalid summary shape.")


def save_diagnostic_archive(archive, *, save=None, savepath=None, overwrite=None):
    """Optionally save one self-contained .npy file; disabled unless save=True.

    Defaults come from setup. Existing targets are protected unless overwrite=True.
    Write a temporary file in the same directory, then publish only after success;
    an interrupted/failed write does not replace an existing completed archive.
    Use store_diagnostic_members to update values, and rerun aggregation after
    ANY direct array edits. This saver checks structure, not numerical freshness.
    """
    settings = archive["settings"]
    enabled = settings["save"] if save is None else save
    if not enabled:
        print("Save disabled; results remain in memory.")
        return None
    _check_completed_diagnostic_archive(archive)
    destination = settings["savepath"] if savepath is None else savepath
    if destination is None:
        raise ValueError("Provide savepath ending in .npy when saving.")
    path = Path(destination).expanduser()
    replace = settings["overwrite"] if overwrite is None else overwrite
    if path.suffix != ".npy":
        raise ValueError("savepath must be a .npy FILE, not a directory.")
    if path.exists() and not replace:
        raise FileExistsError(f"Archive already exists: {path}. Choose a new name or set overwrite=True.")
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = dict(archive)
    payload["metadata"] = {**archive["metadata"], "saved_utc": datetime.now(timezone.utc).isoformat()}
    temporary_path = None
    try:
        with tempfile.NamedTemporaryFile(dir=path.parent, prefix=f".{path.stem}_", suffix=".tmp", delete=False) as handle:
            temporary_path = Path(handle.name)
            np.save(handle, payload, allow_pickle=True)
            handle.flush()
            os.fsync(handle.fileno())
        if replace:
            os.replace(temporary_path, path)
        else:
            # Same-filesystem hard link publishes the finished file without any
            # overwrite race. If another file appeared meanwhile, this raises.
            os.link(temporary_path, path)
    finally:
        if temporary_path is not None and temporary_path.exists():
            temporary_path.unlink()       # Remove only this call's exact temporary file.
    archive["metadata"]["saved_utc"] = payload["metadata"]["saved_utc"]
    print(f"Saved diagnostic archive: {path}")
    return path


def load_diagnostic_archive(savepath):
    """Load a completed archive saved above. ONLY load files you created/trust.

    .npy dictionaries use pickle, which is not safe for untrusted files. Checks
    after loading verify structure; they cannot make untrusted pickle safe.
    Loading does not recompute means or change existing notebook variables.
    """
    archive = np.load(Path(savepath).expanduser(), allow_pickle=True).item()
    _check_completed_diagnostic_archive(archive)
    return archive


In [ ]:
# diagnostics["member_values"][experiment][quantity][model]  # (member, ...)
# diagnostics["model_means"][experiment][quantity]           # (model, ...)
# diagnostics["MMM"][experiment][quantity]   

In [ ]:
# %% 1. Register your quantities and create EMPTY output structures
# In the supplied notebook, the preceding helper cell defines these functions.
# Elsewhere, import them from diagnostic_archive_helpers.py (put it on your path):
# from diagnostic_archive_helpers import (setup_diagnostic_archive, store_diagnostic_members,
#     copy_diagnostic_subset, aggregate_diagnostic_archive, save_diagnostic_archive, load_diagnostic_archive)
import numpy as np
from pathlib import Path

# Each quantity has its OWN per-member shape. These two placeholders demonstrate
# a map and scalar; the commented entries show a series and higher-dimensional array.
# dims excludes the member axis. shape is optional; otherwise the first stored
# model determines the shape required for that quantity everywhere in this archive.
quantity_info = {
    "my_map": {"kind": "map", "dims": ("lat", "lon"), "units": "FILL IN",
               "meaning": "FILL IN: your map diagnostic", "apply_mask": True},
    "my_scalar": {"kind": "scalar", "dims": (), "shape": (), "units": "FILL IN",
                  "meaning": "FILL IN: your scalar diagnostic", "apply_mask": False},
    # "my_series": {"kind": "series", "dims": ("time",), "units": "...", "meaning": "..."},
    # "my_array": {"kind": "array", "dims": ("lag", "lat", "lon"), "units": "...",
    #              "meaning": "...", "apply_mask": True},
}

# Reuse the core archive's explicit experiment/model/member inventory. For a new
# project, supply your own dictionaries in the same format. Coordinates are labels
# and validation aids, not instructions to regrid or align arbitrary arrays.
experiments = ["historical", "historical_matched", "amip_hist"]
coordinates = {key: overlap_coordinates[key] for key in ["lat", "lon", "time"]}
# coordinates["lag"] = np.arange(-12, 13)     # If you add a lag-indexed quantity.
analysis_mask = None                         # Optional 2-D Boolean or 1/NaN mask.
# analysis_mask = overlap_coordinates["local_support"]
# A mask affects only the FINAL STORED arrays of apply_mask=True fields.
# If you need region-specific calculations (e.g. a regional time series), apply
# the region to your INPUTS and weights yourself in the calculation block below.

variability_archive_dir = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412")
diagnostics = setup_diagnostic_archive(
    quantity_info, overlap_model_names, overlap_member_ids,
    experiments=experiments, coordinates=coordinates, mask=analysis_mask,
    save=False, savepath=variability_archive_dir / "derived" / "my_diagnostics.npy", overwrite=False,
    metadata={
        "title": "FILL IN: analysis name", "purpose": "FILL IN: question you are exploring",
        "period": "2000-03 through 2014-12", "source_archive": str(variability_archive_dir),
        "preprocessing": "FILL IN: inherited preprocessing and any NEW processing you do",
        "formula": "FILL IN, or put separate formulas under each quantity",
        "notes": "FILL IN: units, signs, missing-value rules, interpretation/caveats",
    },
)
# Optional extra dictionaries/arrays are also saved in the same archive; helpers
# leave them alone. No spatial weights are applied automatically to your results.
# diagnostics["weights"] = np.asarray(overlap_coordinates["base_weights"]).copy()
# diagnostics["observations"] = {}       # Fill with your own reference diagnostics if useful.

# Optional reuse: only skip a subset if its source is ALSO selected. This is valid
# for member diagnostics independent of which other models belong to the group.
# Set False if your calculations themselves depend on the experiment's population.
reuse_historical_matched = True
reuse_matched = reuse_historical_matched and {"historical", "historical_matched"}.issubset(diagnostics["experiments"])
calculate_experiments = [e for e in diagnostics["experiments"] if not (reuse_matched and e == "historical_matched")]


In [ ]:
# %% 2. YOUR calculations inside the visible experiment -> model -> member loops
# This cell deliberately stops until you fill 'here'. Nothing scientific is
# calculated automatically. Rerun setup when changing labels/quantities/masks;
# rerunning it resets ONLY this new diagnostics archive, not your source archive.
for experiment in calculate_experiments:
    for model in diagnostics["model_names"][experiment]:
        member_labels = diagnostics["member_ids"][experiment][model]
        collected = {quantity: [] for quantity in diagnostics["quantity_info"]}
        print(f"{experiment} / {model}: {len(member_labels)} members")

        # OPTIONAL MODEL-LEVEL PREPARATION: open files or retrieve all members once.
        # With raw files, check their member labels against member_labels; reorder
        # explicitly if needed. Close any files you open after this model's loop.
        # model_input = overlap_member_values[experiment]["sigma_Ti"][model]

        for member_index, member_id in enumerate(member_labels):
            here = {}        # One scalar/array for EACH registered quantity, for THIS member.

            # ==================== YOUR CALCULATIONS GO HERE ====================
            # Minimal structural examples; uncomment/change BOTH if keeping the
            # two default quantity names. Update their units/meaning above too.
            # local_sd = overlap_member_values[experiment]["sigma_Ti"][model][member_index]
            # here["my_map"] = local_sd**2       # Example: a temporal-variance map, K^2.
            # here["my_scalar"] = overlap_member_values[experiment]["sigma_R"][model][member_index]
            # Any additional fields: here["your_registered_name"] = your_array
            # No member axis here: map=(lat,lon), series=(time,), scalar=(), etc.
            # ==================================================================

            # Require every field explicitly. An all-NaN numeric result is allowed
            # when scientifically undefined; an accidentally missing field is not.
            if set(here) != set(diagnostics["quantity_info"]):
                raise ValueError("Fill 'here' with exactly the registered quantity keys before running this loop.")
            for quantity in diagnostics["quantity_info"]:
                collected[quantity].append(here[quantity])

        # Stack ONCE per model: the new leading axis follows member_labels exactly.
        # Store checks member count/array shape and applies any opted-in output mask.
        for quantity in diagnostics["quantity_info"]:
            store_diagnostic_members(diagnostics, experiment, quantity, model, np.stack(collected[quantity], axis=0))
        # Close/release your raw model input here if you opened one above.

# Reuse full historical results for the matched model/member inventory. Actual
# member IDs are checked and reordered if necessary; copied arrays are independent.
if reuse_matched:
    copy_diagnostic_subset(diagnostics, source="historical", target="historical_matched", overwrite=True)


In [ ]:
# %% 3. Aggregate finished member diagnostics, then inspect the familiar structures
# No transformations here: if you want mean variance, compute variance per member;
# if you want mean log ratio, compute log ratios per member. This always takes
# finite arithmetic member means, then equal-model means of those diagnostics.
diagnostics = aggregate_diagnostic_archive(diagnostics)

# Convenient SHORT ALIASES into the same archive, not additional data copies.
# Refresh these aliases after re-running aggregation or replacing diagnostics.
member_values = diagnostics["member_values"]        # [experiment][quantity][model] -> (member, ...)
model_means = diagnostics["model_means"]            # [experiment][quantity] -> (model, ...)
MMM = diagnostics["MMM"]                            # [experiment][quantity] -> (...)
model_n_valid = diagnostics["model_n_valid"]        # Finite member counts, matching model_means.
MMM_n_valid = diagnostics["MMM_n_valid"]            # Finite model counts, matching MMM.

# Example lookups (replace the field/model names as needed):
# member_values["amip_hist"]["my_map"][model]
# i = diagnostics["model_names"]["historical_matched"].index("TaiESM1")
# model_means["historical_matched"]["my_map"][i]
# MMM["amip_hist"]["my_map"]


In [ ]:
# %% 4. Save after inspection (uses save/path/overwrite settings from setup)
# A single .npy contains the values, means, counts, labels, coordinates, and metadata.
# Setup defaults to save=False: this call then does NOTHING to disk.
saved_path = save_diagnostic_archive(diagnostics)
# Explicit one-off alternative, without changing the setup settings:
# saved_path = save_diagnostic_archive(diagnostics, save=True, savepath="./my_diagnostics.npy", overwrite=False)


In [ ]:
# %% 5. Load-only example: run the helper cell/import first; no source data needed
# Keep this opt-in so Run All does not reload an older file over fresh calculations.
from pathlib import Path

load_saved_diagnostics = False
if load_saved_diagnostics:
    load_path = Path("/scratch/leiff/MCA/amip_clean/data/maps_2000_2014_covImp/overlap_core_200003_201412/derived/my_diagnostics.npy")
    loaded_diagnostics = load_diagnostic_archive(load_path)   # Only load files you created/trust.
    loaded_member_values = loaded_diagnostics["member_values"]
    loaded_model_means = loaded_diagnostics["model_means"]
    loaded_MMM = loaded_diagnostics["MMM"]
    loaded_metadata = loaded_diagnostics["metadata"]
